In [ ]:
import os, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc
)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore')

print("GPU available:", torch.cuda.is_available())

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


GPU available: False


In [ ]:
#Cell 3 — Load Excel files (local folder, no Drive)
dataset_path = "Cricket/Dataset"  # adjust
output_path  = "Cricket/Results" # adjust
os.makedirs(output_path, exist_ok=True)

df = pd.concat(excel_dfs.values(), ignore_index=True)
print(f"\nCombined shape: {df.shape}")

Loaded 'Cricket - Copy.xlsx' with shape (2979, 2)
Loaded 'Cricket Sentiment Analysis.xlsx' with shape (149, 2)
Loaded 'cricket_comments_balanced.xlsx' with shape (17652, 2)
Loaded 'Facebook Cricket Comments.xlsx' with shape (5000, 2)

Combined shape: (25780, 2)


In [ ]:
#Cell 5 — Clean text
def preprocess_bangla(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[a-zA-Z]', '', text)
    text = re.sub(r'[0-9০-৯]', '', text)
    text = re.sub(r'[^\u0980-\u09FF\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
df['cleaned_text'] = df[TEXT_COLUMN].apply(preprocess_bangla)
df = df[df['cleaned_text'].str.strip() != ''].reset_index(drop=True)
print(f'Samples after cleaning: {len(df)}')

Samples after cleaning: 25779


In [ ]:
#Cell 6 — Label encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df[LABEL_COLUMN])

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}


In [ ]:
#Cell 7 — 70/15/15 split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f'Train: {len(train_df)} ({len(train_df)/len(df):.1%})')
print(f'Val:   {len(val_df)} ({len(val_df)/len(df):.1%})')
print(f'Test:  {len(test_df)} ({len(test_df)/len(df):.1%})')

Train: 18045 (70.0%)
Val:   3867 (15.0%)
Test:  3867 (15.0%)


In [ ]:
# Cell 11 — Train Bangla-BERT-base
results['BanglaBERT-base'] = train_transformer(
    model_name="sagorsarker/bangla-bert-base",
    run_name="BanglaBERT-base",
    use_fast=False,     # Changed to False to avoid BNLP tokenizer conflicts
    epochs=3,           # Matches your optimal 3-epoch setup for the 70/15/15 split
    batch_size=16,      # Consistent batch size for fair comparison
    eval_batch_size=16
)
print(results['BanglaBERT-base']['report'])


Training: BanglaBERT-base (sagorsarker/bangla-bert-base)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/18045 [00:00<?, ? examples/s]

Map:   0%|          | 0/3867 [00:00<?, ? examples/s]

Map:   0%|          | 0/3867 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.346800,0.356436,0.838635,0.845121
2,0.288700,0.359094,0.841479,0.849749
3,0.198200,0.447129,0.843289,0.851100


Saving vocabulary to D:/Mithila_CSE_30_A/Cricket/Results\BanglaBERT-base\vocab.txt: vocabulary indices are not consecutive. Please check that the vocabulary is not corrupted!
Saving vocabulary to D:/Mithila_CSE_30_A/Cricket/Results\BanglaBERT-base\vocab.txt: vocabulary indices are not consecutive. Please check that the vocabulary is not corrupted!
Saving vocabulary to D:/Mithila_CSE_30_A/Cricket/Results\BanglaBERT-base\vocab.txt: vocabulary indices are not consecutive. Please check that the vocabulary is not corrupted!
Saving vocabulary to D:/Mithila_CSE_30_A/Cricket/Results\BanglaBERT-base\vocab.txt: vocabulary indices are not consecutive. Please check that the vocabulary is not corrupted!
Saving vocabulary to D:/Mithila_CSE_30_A/Cricket/Results\BanglaBERT-base\vocab.txt: vocabulary indices are not consecutive. Please check that the vocabulary is not corrupted!
Saving vocabulary to D:/Mithila_CSE_30_A/Cricket/Results\BanglaBERT-base\vocab.txt: vocabulary indices are not consecutive. P

              precision    recall  f1-score   support

    negative       0.77      0.77      0.77      1364
     neutral       0.99      0.96      0.98      1137
    positive       0.78      0.79      0.78      1366

    accuracy                           0.84      3867
   macro avg       0.85      0.84      0.84      3867
weighted avg       0.84      0.84      0.84      3867



In [ ]:
# Cell 12 — Train XLM-RoBERTa
results['XLM-RoBERTa'] = train_transformer(
    model_name="xlm-roberta-base",
    run_name="XLM-RoBERTa",
    use_fast=True,      # Kept True as XLM-RoBERTa supports fast tokenization perfectly
    epochs=3,           # Kept at 3 to keep the benchmark comparison fair
    batch_size=16,      # Consistent batch size
    eval_batch_size=16
)
print(results['XLM-RoBERTa']['report'])


Training: XLM-RoBERTa (xlm-roberta-base)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/18045 [00:00<?, ? examples/s]

Map:   0%|          | 0/3867 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative       0.78      0.82      0.80      1364
     neutral       0.99      0.96      0.98      1137
    positive       0.81      0.78      0.79      1366

    accuracy                           0.85      3867
   macro avg       0.86      0.85      0.86      3867
weighted avg       0.85      0.85      0.85      3867

